# Explore Umar's pipeline dataset

**Stage 1: understand the files before choosing a method.**

The path is configured for your Windows computer: `F:\Pipeline Data\pipeline data`.
Keep `pipeline_audit.py` next to this notebook. Run the cells in order.
The screenshot suggests gas/water and 13/18 bar conditions; these are unverified
filename hints. We have not read your actual dataset or measured any model results.

This notebook inventories the data, saves small previews, identifies data-quality
issues and produces a report you can upload here. It does not guess labels,
sample rate, sensor roles, leak positions or experimental groups. It reads raw
files without modifying them. All initial statistics describe **head samples**.


## 1. Environment

Install the packages once in the same Python environment as this notebook.
Uncomment the command below if needed, then restart the kernel after installation.
The README contains the full Windows setup commands. No GPU is required.


In [ ]:
# %pip install -r requirements.txt


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
try:
    from IPython.display import display
except ImportError:
    display = print

from pipeline_audit import run_audit, load_preview, numeric_values, make_bundle
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
print("Imports ready. Notebook folder:", Path.cwd())


## 2. Configure the folder

Keep the `r` before the Windows path string. Output is saved in a sibling
directory, outside the raw dataset. Leave `OVERRIDES` empty for the first audit.
If the folder is unavailable, this notebook must be run on the Windows PC where
that drive is connected. The maximum below limits rows per table, not file size.


In [ ]:
ROOT = Path(r"F:\Pipeline Data\pipeline data")
OUT = ROOT.parent / "pipeline_audit_outputs"
MAX_ROWS = 5_000
MAX_COLUMNS = 128
HASH_FULL_FILES = False  # True checks exact duplicates but reads all file bytes.

# Only add overrides after inspecting real file contents and extensions.
# Example: {"Gas_13Bar.csv": {"header_row": 0, "delimiter": ";", "decimal": ","}}
OVERRIDES = {}

if not ROOT.is_dir():
    raise FileNotFoundError(f"Cannot find {ROOT}. Run locally and check the drive/folder.")
print("Dataset:", ROOT)
print("Outputs:", OUT)


## 3. Run the audit

CSV/TSV/TXT/DAT and Excel worksheets are previewed. MATLAB files receive variable
metadata only; other formats remain visible in the inventory. Each run has a new
output folder. A header is preserved as a data row until you explicitly identify it.

This bounded preview can miss events later in a recording. It cannot establish
that a file is healthy or estimate complete class balance.


In [ ]:
RUN_DIR = run_audit(
    ROOT, OUT, max_rows=MAX_ROWS, max_cols=MAX_COLUMNS,
    plots=True, hash_files=HASH_FULL_FILES, overrides=OVERRIDES,
)
report = json.loads((RUN_DIR / "report.json").read_text(encoding="utf-8"))
inventory = pd.read_csv(RUN_DIR / "inventory.csv")
tables = pd.read_csv(RUN_DIR / "tables.csv")
column_profile = pd.read_csv(RUN_DIR / "column_profile.csv")
issues = pd.read_csv(RUN_DIR / "issues.csv")
display(inventory)


In [ ]:
display(tables)
display(issues if len(issues) else "No parsing issues recorded.")
display(column_profile.head(40))
print("More profiles are saved in:", RUN_DIR / "column_profile.csv")
print("Exact duplicate files:", report["exact_duplicate_file_groups"] if HASH_FULL_FILES else "not checked")


## 4. Inspect one table carefully

Choose the real relative filename and worksheet from the inventory. With
`SELECTED_FILE=None`, the first successfully previewed table is displayed only as
a convenient example; no scientific meaning is assigned to it. `HEADER_ROW=None`
keeps every row. Set it to `0` only if the first record is a confirmed header.
`SKIPROWS` removes a known metadata preamble before the header.

Check whether rows are time samples, independent examples, or transposed trials.
Check units and whether numeric columns are labels, identifiers or measurements.


In [ ]:
SELECTED_FILE = None      # e.g. "Gas_13Bar.csv" ONLY if inventory confirms that name
SELECTED_SHEET = None     # e.g. "Sheet1" for Excel
HEADER_ROW = None         # None = headerless; 0 = first record is a header
SKIPROWS = 0
DELIMITER = None          # auto, or ",", ";", "\t", "whitespace"
ENCODING = None           # auto, or a confirmed encoding such as "utf-8-sig"
DECIMAL = "."            # use "," only for confirmed decimal-comma numbers

selected = SELECTED_FILE
selected_sheet = SELECTED_SHEET
if selected is None and report["tables"]:
    selected = report["tables"][0]["file"]
    selected_sheet = report["tables"][0]["sheet"] or None

df = pd.DataFrame()
parser_info = {}
if selected is None:
    print("No table was previewed. Inspect issues.csv before continuing.")
else:
    df, parser_info = load_preview(
        ROOT / selected, sheet=selected_sheet,
        max_rows=MAX_ROWS, max_cols=MAX_COLUMNS,
        header_row=HEADER_ROW, skiprows=SKIPROWS,
        delimiter=DELIMITER, encoding=ENCODING,
    )
    print("Selected:", selected, selected_sheet)
    display(parser_info)
    display(df.head(15))
    print("Sample shape:", df.shape)


## 5. Confirm sensor and time fields

This section is optional. Leave it blank for a first report. Enter actual column
names shown above, excluding labels, trial IDs and operating-condition metadata.
Do not call a row index 'seconds'. If you confirm a timestamp, specify its unit.
Timestamp checks below describe this sample only; they do not determine the
sensor acquisition rate automatically.


In [ ]:
SENSOR_COLUMNS = []   # e.g. ["pressure_in", "pressure_out"] AFTER confirmation
SENSOR_UNITS = {}     # e.g. {"pressure_in": "bar", "pressure_out": "bar"}
TIME_COLUMN = None
TIME_UNIT = "s"      # "s", "ms", "us", "ns", or "datetime"
FS_HZ = None         # verified sampling frequency; never guess this value
TIME_SERIES_CONFIRMED = False
UNIFORM_SAMPLING_CONFIRMED = False
SINGLE_CONTIGUOUS_RECORDING_CONFIRMED = False

if TIME_COLUMN is not None:
    if TIME_COLUMN not in df:
        raise KeyError(f"Unknown time column: {TIME_COLUMN}")
    if TIME_UNIT == "datetime":
        ts = pd.to_datetime(df[TIME_COLUMN], errors="coerce", utc=True)
        valid = ts.dropna()
        time_s = (ts - valid.iloc[0]).dt.total_seconds() if len(valid) else pd.Series(np.nan, index=df.index)
    else:
        scale = {"s": 1.0, "ms": 1e-3, "us": 1e-6, "ns": 1e-9}[TIME_UNIT]
        time_s, _ = numeric_values(df[TIME_COLUMN], decimal=DECIMAL)
        time_s = time_s * scale
    differences = time_s.diff()
    dt = differences[np.isfinite(differences)]
    positive = dt[dt > 0]
    display({"missing_or_invalid_timestamps": int((~np.isfinite(time_s)).sum()),
             "repeated_adjacent_times": int((dt == 0).sum()),
             "backward_adjacent_times": int((dt < 0).sum()),
             "median_positive_dt_seconds": float(positive.median()) if len(positive) else None,
             "max_positive_dt_seconds": float(positive.max()) if len(positive) else None})
    print("Check gaps, resets and units before treating these rows as one time series.")
else:
    print("Time column not confirmed; plots use row/sample index.")


In [ ]:
if SENSOR_COLUMNS:
    missing_columns = set(SENSOR_COLUMNS) - set(df.columns)
    if missing_columns:
        raise KeyError(f"Columns not in selected table: {missing_columns}")
    signals = pd.DataFrame({col: numeric_values(df[col], decimal=DECIMAL)[0] for col in SENSOR_COLUMNS})
    signals = signals.replace([np.inf, -np.inf], np.nan)
    display(signals.describe().T)
    display(signals.isna().sum().rename("missing_or_non_numeric_or_infinite"))
    shown = SENSOR_COLUMNS[:8]
    fig, axes = plt.subplots(len(shown), 1, figsize=(11, 2.6 * len(shown)), squeeze=False, layout="constrained")
    for ax, col in zip(axes.flat, shown):
        if TIME_COLUMN is not None:
            ax.plot(time_s, signals[col], linewidth=0.8)
            ax.set_xlabel("Time (seconds, supplied time column)")
        else:
            ax.plot(np.arange(len(signals)), signals[col], linewidth=0.8)
            ax.set_xlabel("Row offset in sample")
        ax.set_ylabel(f"{col} [{SENSOR_UNITS.get(col, 'unit unconfirmed')}]")
        ax.grid(alpha=0.2)
    fig.suptitle(f"Confirmed selected channels: {selected}")
    fig.savefig(RUN_DIR / "confirmed_signals.png", dpi=160)
    plt.show()
    plt.close(fig)
else:
    signals = pd.DataFrame()
    print("Sensor columns not confirmed. Use the audit plots only for initial inspection.")


## 6. Optional frequency analysis

Only run this on a single contiguous recording with confirmed uniformly sampled
sensor channels. The code refuses spectra with missing values; deleting gaps or
concatenating trials would corrupt frequency interpretation. It applies Welch's
power spectral density estimate separately to each selected channel. Its default
detrending removes the mean for the spectrum; raw amplitude plots above remain
available. Do not infer leak labels from these plots alone.


In [ ]:
ready_for_psd = (len(signals) > 0 and FS_HZ is not None and TIME_SERIES_CONFIRMED
                 and UNIFORM_SAMPLING_CONFIRMED and SINGLE_CONTIGUOUS_RECORDING_CONFIRMED)
if ready_for_psd:
    if not np.isfinite(FS_HZ) or FS_HZ <= 0:
        raise ValueError("FS_HZ must be a verified positive frequency.")
    if len(signals) < 16:
        raise ValueError("Too few samples for this spectrum preview.")
    if signals.isna().any().any():
        raise ValueError("Missing/non-numeric values found. Select a genuinely contiguous valid segment first.")
    if TIME_COLUMN is not None:
        dt_all = np.diff(time_s.to_numpy(dtype=float))
        if not np.isfinite(dt_all).all() or not np.allclose(dt_all, 1.0 / FS_HZ, rtol=0.01, atol=1e-12):
            raise ValueError("Timestamps do not match the supplied uniform sampling rate within 1%. Inspect time units and precision.")
    shown = SENSOR_COLUMNS[:8]
    fig, axes = plt.subplots(len(shown), 1, figsize=(11, 2.6 * len(shown)), squeeze=False, layout="constrained")
    for ax, col in zip(axes.flat, shown):
        frequencies, power = welch(signals[col].to_numpy(), fs=float(FS_HZ), nperseg=min(1024, len(signals)))
        ax.semilogy(frequencies, np.maximum(power, np.finfo(float).tiny))
        ax.set(xlabel="Frequency (Hz)", ylabel=f"PSD ({SENSOR_UNITS.get(col, 'unit')}²/Hz)", title=col)
        ax.grid(alpha=0.2)
    fig.savefig(RUN_DIR / "confirmed_spectra.png", dpi=160)
    plt.show()
    plt.close(fig)
else:
    print("PSD skipped: confirm channels, sampling rate, time order and a single contiguous uniform recording first.")


## 7. Record the experimental meaning

Open `recording_metadata_template.csv` in the output folder and fill known fields
from the experiment log. Leave unknown entries blank. A row initially represents
a table, **not necessarily one recording**. If several experiments occupy a table,
add rows with data-row bounds. If trials are arranged in columns, describe that
layout in `notes` so we can write an appropriate loader.

Most useful facts:

1. Which column is each sensor, timestamp, operating parameter or label? What are the units?
2. Is a label per sample, window or whole recording? What do its values mean?
3. Which files/segments belong to the same experimental run, day or setup?
4. What are the sampling rate, sensor positions, pipeline length and leak positions/sizes?
5. Is there genuine time order across operating-condition changes? Are there repeated independent runs?
6. Where did the dataset come from, and is there a paper, README or collection protocol?

Never infer healthy/leak labels from the absence or presence of 'Leak' in a filename.
Never interpret file_data1/file_data2 as class labels without the collection protocol.


In [ ]:
metadata_path = RUN_DIR / "recording_metadata_template.csv"
display(pd.read_csv(metadata_path).head(12))
print("Fill this file from your experimental records:", metadata_path)


## 8. Research decision and next exchange

Read `RESEARCH_PLAN.md`. A provisional direction is leak-preserving adaptation
across operating pressures. It is only appropriate if the verified data support
the task, grouped splits and temporal protocol. TARD already addresses the risk
of adapting to faults; simply combining familiar components is not evidence of
novelty. A specific failure mode, a justified mechanism, and rigorous ablations
must establish any new contribution.

Before model development, upload **audit_bundle.zip** and ideally one small
representative original data file. If you edited the metadata template or ran
the optional signal analysis, rerun the next cell to refresh the archive.

The archive includes summaries and small raw-data previews. The code does not
send any files or connect to the internet. No scores on your data are present.


In [ ]:
bundle = make_bundle(RUN_DIR)
print("Upload this report here:", bundle)
print("Also useful: one representative original file and the collection protocol.")
